# Hypothesis Testing: Season 15's Feat-of-Strength makes the early game more decisive of final match outcome.

The feat-of-strength is decided as the following goals, where each team race to get 2/3 first.
1. First team to 3 kills.
2. First team to claim a tower.
3. First team to 3 epic monsters (dragons, void grubs, heralds, barons).

* **Null Hypothesis:** Winrate with feat-of-strength = Winrate without feat-of-strength
* **Alternative Hypothesis:** Winrate with feat-of-strength > Winrate without feat-of-strength
* **Alpha:** 0.05

> *Note: This notebook and its work is only shown here for demonstration, using the full seed source and hand-run calculations. This will not run in the pipeline.*

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm, t

In [2]:
INTERVALS_DIR = '../../data/league/intervals.csv'
MATCHES_DIR = '../../data/league/matches.csv'

In [ ]:
# ----- Raw matches data
S15_16_MATCHES = (pd.read_csv(MATCHES_DIR)
    .assign(season=lambda df: pd.to_numeric(df['game_version'].str[:2]))
    .reset_index(drop=True)
    .query('season == 15 or season == 16')
    .query('game_duration >= 300')
     [['match_id', 'season', 'winning_team']]
)

In [ ]:
# ----- Raw intervals data
TEAM_INTERVALS = (pd.read_csv(INTERVALS_DIR)
    .assign(team_epic_monsters=lambda df: pd.to_numeric(
        df['team_dragons'] + df['team_barons'] + df['team_void_grubs'] + df['team_heralds']
    ))
    [['match_id', 'team', 'minute', 'team_kills', 'team_towers', 'team_epic_monsters']]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
S15_16_TEAM_INTERVALS = (S15_16_MATCHES
    .merge(TEAM_INTERVALS, on='match_id', how='inner')
    .assign(is_winner=lambda df: df['winning_team'] == df['team'])
    [['match_id', 'season', 'team', 'minute', 'team_kills', 'team_towers', 'team_epic_monsters', 'is_winner']]
    .sort_values(['match_id', 'minute'])
 )

## Prepare data for analysis
1. Calculate which team has the first feat-of-strength for the entire match
2. Aggregate win bernoulli array into 'with feat' and 'without feat'
(e.g. with feat 0 0 0 1 1 / w/out feat 0 0 1 1 0)

In [ ]:
def first_feat_of_strength(
    df: pd.DataFrame,
    season: int = 15,
) -> pd.DataFrame:
    df = df.loc[df['season'] == season]
    milestone_minute = (
        df.assign(
            kills_min=np.where(df["team_kills"] >= 3, df["minute"], np.nan),
            tower_min=np.where(df["team_towers"] >= 1, df["minute"], np.nan),
            epic_min=np.where(df["team_epic_monsters"] >= 3, df["minute"], np.nan),
        )
        .groupby(["match_id", "team"], as_index=False)
        .agg(
            kills_min=("kills_min", "min"),
            tower_min=("tower_min", "min"),
            epic_min=("epic_min", "min"),
        )
    )

    # 2. For each team, the minute they'd secure FoS = 2nd-smallest of the three
    #    milestone minutes (i.e. when the SECOND milestone was reached).
    milestone_cols = ["kills_min", "tower_min", "epic_min"]

    def second_milestone_minute(row):
        vals = sorted(v for v in row[milestone_cols] if pd.notna(v))
        return vals[1] if len(vals) >= 2 else np.nan

    milestone_minute["fos_minute"] = milestone_minute.apply(second_milestone_minute, axis=1)

    # 3. Pivot so each match has one row with both teams' fos_minute side by side.
    pivoted = milestone_minute.pivot(index="match_id", columns="team", values="fos_minute")
    pivoted = pivoted.rename(columns={
        "BLUE": "blue_fos_minute",
        "RED": "red_fos_minute"
    })
    pivoted = pivoted.reset_index()

    for col in ["blue_fos_minute", "red_fos_minute"]:
        if col not in pivoted.columns:
            pivoted[col] = np.nan

    def determine_team_with_feat(row):
        blue_m, red_m = row["blue_fos_minute"], row["red_fos_minute"]
        blue_ok, red_ok = pd.notna(blue_m), pd.notna(red_m)

        if not blue_ok and not red_ok:
            return pd.Series(["NO_QUALIFIER", None])
        if blue_ok and not red_ok:
            return pd.Series(["RESOLVED", "BLUE"])
        if red_ok and not blue_ok:
            return pd.Series(["RESOLVED", "RED"])
        # Both reached 2 milestones -- earlier minute wins; true tie -> unresolved.
        if blue_m < red_m:
            return pd.Series(["RESOLVED", "BLUE"])
        elif red_m < blue_m:
            return pd.Series(["RESOLVED", "RED"])
        else:
            return pd.Series(["TIE", None])

    pivoted[["drop_reason", "team_with_feat"]] = pivoted.apply(determine_team_with_feat, axis=1)

    # 4. Attach winning_team (constant per match_id -- take from is_winner flag).
    winner = (
        df[df["is_winner"]]
        .drop_duplicates(subset="match_id")[["match_id", "team"]]
        .rename(columns={"team": "winning_team"})
    )

    result = pivoted.merge(winner, on="match_id", how="left")
    result = result[["match_id", "winning_team", "team_with_feat", "drop_reason"]]

    # ----- 5. Report and drop matches where team_with_feat could not be resolved, broken down by reason.
    total_matches = result["match_id"].nunique()

    n_no_qualifier = int((result["drop_reason"] == "NO_QUALIFIER").sum())
    n_tie = int((result["drop_reason"] == "TIE").sum())
    n_unresolved = n_no_qualifier + n_tie

    if n_unresolved > 0:
        pct = n_unresolved / total_matches * 100
        print(
            f"[first_feat_of_strength] Dropped {n_unresolved:,} / {total_matches:,} "
            f"match_id ({pct:.2f}%) with no clear team_with_feat:"
        )
        print(
            f"    - {n_no_qualifier:,} match_id where neither team ever secured "
            f"2 of the 3 milestones"
        )
        print(
            f"    - {n_tie:,} match_id where both teams reached their 2nd milestone "
            f"at the same sampled minute (unresolvable tie at 5-minute granularity)"
        )
    else:
        print(
            f"[first_feat_of_strength] All {total_matches:,} match_id resolved a "
            f"clear team_with_feat. None dropped."
        )

    match_level = (result
        .loc[result["drop_reason"] == "RESOLVED"]
        .reset_index(drop=True)
        .drop(columns="drop_reason")
    )

    # ----- 6. Reshape match-grain (1 row/match, 2 team columns folded in) into
    #          team-grain (1 row per match_id x team, with has_feat/is_winner booleans).
    team_level = (
        match_level[["match_id"]]
        .merge(pd.DataFrame({"team": ["BLUE", "RED"]}), how="cross")
        .merge(match_level, on="match_id", how="left")
    )
    team_level["has_feat"] = team_level["team"] == team_level["team_with_feat"]
    team_level["is_winner"] = (team_level["team"] == team_level["winning_team"]).astype('int64')

    return team_level[["match_id", "team", "has_feat", "is_winner"]]

In [ ]:
S15_FOS_SUMMARIES = first_feat_of_strength(S15_16_TEAM_INTERVALS, season=15)
S16_FOS_SUMMARIES = first_feat_of_strength(S15_16_TEAM_INTERVALS, season=16)

[first_feat_of_strength] Dropped 280 / 1,583 match_id (17.69%) with no clear team_with_feat:
* 10 match_id where neither team ever secured 2 of the 3 milestones
* 270 match_id where both teams reached their 2nd milestone at the same sampled minute (unresolvable tie at 5-minute granularity)

[first_feat_of_strength] Dropped 5,603 / 37,811 match_id (14.82%) with no clear team_with_feat:
* 144 match_id where neither team ever secured 2 of the 3 milestones
* 5,459 match_id where both teams reached their 2nd milestone at the same sampled minute (unresolvable tie at 5-minute granularity)

## Exploratory Data Analysis
1. Explore distribution of win rate between team that has feat of strength vs those that do not.

In [ ]:
def barplot_fos(season: int = 15) -> tuple:
    # ----- Aggregate data
    data = S15_FOS_SUMMARIES if season == 15 else S16_FOS_SUMMARIES
    data = (data
        .groupby(['team', 'has_feat'], as_index=False)
        .agg(win_count=('is_winner', 'sum'))
    )

    # ----- Visualize
    fig, ax = plt.subplots(figsize=(10,6), dpi=200)
    sns.barplot(
        data=data,
        x='has_feat',
        y='win_count',
        hue='team',
        ax=ax
    )

    ax.set_title(f'Win count for teams with FoS (Feat-of-Strength) versus teams w/out (season {season})')
    ax.set_xlabel('Has FoS (Feat-of-Strength)')
    ax.set_ylabel('Win counts')

    return fig, ax

In [ ]:
def vis_hist(arr: np.ndarray, x_lb: str) -> tuple:
    fig, ax = plt.subplots(figsize=(10,6), dpi=200)
    sns.histplot(x=arr, ax=ax, stat='probability')

    ax.set_title(f'Distribution of {x_lb}')
    ax.set_xlabel(x_lb)
    ax.set_ylabel('Probability')

    return fig, ax

In [ ]:
def bernoulli_arr_stats(
    arr: np.ndarray,
    ci: float = 0.95,
) -> dict[str, int | float]:
    n = int(arr.shape[0])
    p_ = float(np.sum(arr) / n)
    se = float(np.sqrt(
        p_ * (1-p_) / n
    ))

    zcrit = norm.ppf(1 - (1 - ci) / 2)
    lci = float(p_ - (zcrit * se))
    uci = float(p_ + (zcrit * se))

    return {'n': n, 'p^': p_, 'se': se, 'ci': (lci, uci)}

In [ ]:
def diff_hypo_arr_stats(
    a_arr: np.ndarray,
    b_arr: np.ndarray,
    alpha: float = 0.05,
    mode: str = 'twotail',
    offset_loc: float = 0,
) -> dict[str, float]:
    a = bernoulli_arr_stats(a_arr)
    b = bernoulli_arr_stats(b_arr)

    # ----- Descriptive
    n = int(a['n'] + b['n'])
    p_ = float(a['p^'] - b['p^'])
    se = float(np.sqrt(
        a['se']**2 + b['se']**2
    ))

    # ----- Inferential
    z = (p_ - offset_loc)/ se
    p, lci, uci = 0, 0, 0
    match mode:
        case 'twotail':
            crit = norm.ppf(1 - alpha / 2)
            p = norm.sf(abs(z)) * 2
            lci = p_ - (crit * se)
            uci = p_ + (crit * se)
        case 'ge':
            crit = norm.ppf(1 - alpha)
            p = norm.sf(z)
            lci = p_ - (crit * se)
            uci = np.inf
        case 'le':
            crit = norm.ppf(alpha)
            print(crit)
            p = norm.cdf(z)
            lci = np.inf
            uci = p_ + (crit * se)
        case _: raise ValueError(f'Mode can only be one of ["twotail", "ge", "le"], got {mode}')

    return {
        'n': n,
        'p^': p_,
        'se': se,
        'p': p,
        'ci': (lci, uci)
    }

In [ ]:
def run(
    season: int = 15,
    team: str = None,
    mode: str = 'twotail',
    alpha: float = 0.05,
    show_plots: bool = False,
    offset_loc: float = 0,
) -> pd.DataFrame:
    src = S15_FOS_SUMMARIES if season == 15 else S16_FOS_SUMMARIES
    if team is not None:
        src = src.query('team == @team')

    # ----- 01. Prepare sample array
    fos_arr = src.loc[lambda df: df['has_feat'], 'is_winner'].to_numpy()
    no_fos_arr = src.loc[lambda df: ~df['has_feat'], 'is_winner'].to_numpy()

    # ----- 02. Visualize distribution per array
    if show_plots:
        vis_hist(fos_arr, 'Win count with feat')
        vis_hist(no_fos_arr, 'Win count without feat')

    # ----- 03. Calculate statistics per array
    fos = bernoulli_arr_stats(fos_arr)
    no_fos = bernoulli_arr_stats(no_fos_arr)

    # ----- 03. Calculate diff statistics
    diff = diff_hypo_arr_stats(
        a_arr=fos_arr,
        b_arr=no_fos_arr,
        mode=mode,
        alpha=alpha,
        offset_loc=offset_loc,
    )

    return pd.DataFrame({
        'grp': ['fos', 'no_fos', 'diff'],
        'n': [fos['n'], no_fos['n'], diff['n']],
        'p^': [fos['p^'], no_fos['p^'], diff['p^']],
        'se': [fos['se'], no_fos['se'], diff['se']],
        'lower_ci': [fos['ci'][0], no_fos['ci'][0], diff['ci'][0]],
        'upper_ci': [fos['ci'][1], no_fos['ci'][1], diff['ci'][1]],
        f'mode={mode} | offset={offset_loc}': [f'alpha={alpha}', f'p={diff['p']}', f'reject_null={diff['p'] <= alpha}']
    })

In [ ]:
run(season=16, mode='ge', alpha=0.05, offset_loc=0.10)

| grp | n | p̂ | se | lower_ci | upper_ci |
|---|---|---|---|---|---|
| fos | 32208 | 0.567933 | 0.002760 | 0.562524 | 0.573343 |
| no_fos | 32208 | 0.432067 | 0.002760 | 0.426657 | 0.437476 |
| diff | 64416 | 0.135867 | 0.003904 | 0.129446 | inf |

**Test parameters:** mode = `ge`, offset = `0.1`, alpha = `0.05`

**Result:** p = 1.995 × 10⁻²⁰, reject_null = `True`

In [ ]:
run(season=15, mode='ge', alpha=0.05, offset_loc=0.10)

| grp | n | p̂ | se | lower_ci | upper_ci |
|---|---|---|---|---|---|
| fos | 1303 | 0.588642 | 0.013632 | 0.561923 | 0.615360 |
| no_fos | 1303 | 0.411358 | 0.013632 | 0.384640 | 0.438077 |
| diff | 2606 | 0.177283 | 0.019279 | 0.145573 | inf |

**Test parameters:** mode = `ge`, offset = `0.1`, alpha = `0.05`

**Result:** p = 3.052 × 10⁻⁵, reject_null = `True`

## Summary

Comparing Season 15 (with feat-of-strength) to Season 16 (without), the early-game advantage in winrate (diff p) shrank from about 18% to about 14% (both diff p statistically significant). The fact that with or without FoS (feat-of-strength) tangible buffs, the win rate remains above 50% for any team in the lead (by securing the FoS objectives) suggest that:
1. FoS amplifies an existing snowball effect that already decides match outcomes.
2. This amplifying effect accounts for a ~4% increase in win probability.

Even though FoS only shifts the win probability by about 4%, players still look back on FoS as something that makes the early game very stressful, like losing it seals the game. This perception is probably embedded by the fact that once you lose the race for it, that's it, there's no second chance to win it back later in the match. The numbers tell a calmer story: teams that lose FoS still go on to win roughly 4 times out of 10.

Future similar features can give a more balanced perception in players probably by encouraging tug-of-war like recoverability.